## Elastic Net only 

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
from matplotlib.ticker import PercentFormatter
from matplotlib.patches import Patch
import pandas as pd
import shap
import numpy as np
import json
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import ElasticNet
import warnings
warnings.filterwarnings('ignore')


# Load and prepare data
df_featimp = pd.read_csv("./data/feature_importance.csv")
df_featimp["pct_increase_in_error"] = 100 * (df_featimp["permuted_performance"] - df_featimp["baseline"]) / df_featimp["baseline"]

# Define columns and mappings
OOS_PRED_COLUMNS_PUNISHPARAMS = [
    'CONFIG_playerCount', 'CONFIG_numRounds', 'CONFIG_showNRounds', 'CONFIG_MPCR',
    'CONFIG_allOrNothing', 'CONFIG_chat', 'CONFIG_defaultContribProp', 'CONFIG_rewardExists',
    'CONFIG_showOtherSummaries', 'CONFIG_showPunishmentId', 'CONFIG_punishmentCost',
    'CONFIG_punishmentTech'
]

# Add the baseline efficiency feature
ALL_FEATURES = OOS_PRED_COLUMNS_PUNISHPARAMS + ['control_itt_efficiency']

DICT_COLUMN_LABELS = {
    'CONFIG_playerCount': 'Group Size',
    'CONFIG_numRounds': 'Game Length',
    'CONFIG_showNRounds': 'Horizon Knowledge',
    'CONFIG_MPCR': 'Return Rate (MPCR)',
    'CONFIG_allOrNothing': 'Contribution Type',
    'CONFIG_chat': 'Communication',
    'CONFIG_defaultContribProp': 'Contribution Framing',
    'CONFIG_rewardExists': 'Reward',
    'CONFIG_showOtherSummaries': 'Peer Outcome Visibility',
    'CONFIG_showPunishmentId': 'Actor Anonymity',
    'CONFIG_punishmentCost': 'Peer Incentive Cost',
    'CONFIG_punishmentTech': 'Punishment Technology'
}

feature_info = {
    '# of players': {'new_name': 'Group Size', 'category': 'Game Structure'},
    '# of rounds': {'new_name': 'Game Length', 'category': 'Game Structure'},
    'Visibility of # of rounds': {'new_name': 'Horizon Knowledge', 'category': 'Game Structure'},
    'Marginal per capita return': {'new_name': 'Return Rate (MPCR)', 'category': 'Game Structure'},
    '"All or Nothing" contributions': {'new_name': 'Contribution Type', 'category': 'Contribution Structure'},
    'Default contribution vs withdrawal': {'new_name': 'Contribution Framing', 'category': 'Contribution Structure'},
    'Ability to chat': {'new_name': 'Communication', 'category': 'Social Information'},
    'Peer outcome visibility': {'new_name': 'Peer Outcome Visibility', 'category': 'Social Information'},
    'Punisher/rewarder anonymity': {'new_name': 'Actor Anonymity', 'category': 'Social Information'},
    'Ability to reward': {'new_name': 'Reward', 'category': 'Incentive Mechanisms'},
    'Punishment cost': {'new_name': 'Peer Incentive Cost', 'category': 'Incentive Mechanisms'},
    'Punishment effectiveness': {'new_name': 'Punishment Technology', 'category': 'Incentive Mechanisms'}
}

# Load learning data
df_paired_learn = pd.read_csv("./data/df_paired_learn.csv")

# Define InteractionTransformer
class InteractionTransformer(BaseEstimator, TransformerMixin):
    def __init__(self):
        pass
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X = X.to_numpy()
        
        n_features = X.shape[1]
        interaction_terms = []
        for i in range(n_features):
            for j in range(i+1, n_features):
                interaction_terms.append(X[:, i] * X[:, j])
        
        result = np.column_stack([X] + interaction_terms)
        
        return result

# Load HPO params
prereg_hpo_params = json.load(open("./data/prereg_hpo_params_09202024.json", "r"))

# Setup and fit elastic net model using ALL_FEATURES
elastic_prereg = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("interactions", InteractionTransformer()),
    ("estimator", ElasticNet(
        alpha=prereg_hpo_params["ELASTIC"]["alpha"], 
        l1_ratio=prereg_hpo_params["ELASTIC"]["l1_ratio"],
        random_state=prereg_hpo_params["ELASTIC"]["random_seed"]))
])
elastic_prereg.fit(
    X=df_paired_learn[ALL_FEATURES], 
    y=df_paired_learn["treatment_itt_efficiency"]
)

# Calculate SHAP values with ALL_FEATURES (including control_itt_efficiency)
background_data = df_paired_learn[ALL_FEATURES].astype(float).values
masker = shap.maskers.Independent(background_data)

def model_wrapper(x):
    if len(x.shape) == 1:
        x = x.reshape(1, -1)
    return elastic_prereg.predict(x)

shap_explainer = shap.Explainer(
    model=model_wrapper,
    masker=masker,
    feature_names=ALL_FEATURES
)
shap_values = shap_explainer(df_paired_learn[ALL_FEATURES].astype(float).values)

# Style setup
plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
matplotlib.rcParams['font.sans-serif'] = ['Arial']
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['axes.facecolor'] = 'white'
matplotlib.rcParams['figure.facecolor'] = 'white'
matplotlib.rcParams['grid.alpha'] = 0.0

# Define categories and colors
category_colors = {
    'Game Structure': '#2166AC',         
    'Contribution Structure': '#92C5DE',    
    'Social Information': '#D6604D',        
    'Incentive Mechanisms': '#F4A582'       
}

# Create figure
width_mm = 183
height_mm = 120
width_inches = width_mm / 25.4
height_inches = height_mm / 25.4

fig = plt.figure(figsize=(width_inches, height_inches), dpi=300)
gs = plt.GridSpec(1, 2, width_ratios=[1, 1], wspace=0.1)
ax_left = fig.add_subplot(gs[0])
ax_right = fig.add_subplot(gs[1])

# Panel A: Feature Importance Plot
df_plot = df_featimp.query("model == 'E-net'").copy()
df_plot['new_name'] = df_plot['feature'].map({k: v['new_name'] for k, v in feature_info.items()})
df_plot['category'] = df_plot['feature'].map({k: v['category'] for k, v in feature_info.items()})

# Get the order based on the mean values
means = df_plot.groupby('new_name')['pct_increase_in_error'].mean()
ordered_features = means.sort_values(ascending=False).index

# Create bar plot
bars = sns.barplot(
    x="pct_increase_in_error",
    y="new_name",
    data=df_plot,
    order=ordered_features,
    ax=ax_left,
    estimator='mean',
    errorbar=('ci', 68),
    capsize=0.2,
    errwidth=1
)

# Color the bars according to category
for i, feature in enumerate(ordered_features):
    category = df_plot[df_plot['new_name'] == feature]['category'].iloc[0]
    bars.patches[i].set_facecolor(category_colors[category])

# Add reference line
ax_left.axvline(x=0, 
                linestyle="--", 
                color="#666666",
                linewidth=0.8,
                zorder=0)

# Style Panel A
ax_left.spines['top'].set_visible(False)
ax_left.spines['right'].set_visible(False)
ax_left.spines['left'].set_linewidth(0.5)
ax_left.spines['bottom'].set_linewidth(0.5)
ax_left.tick_params(axis="both", which='major', labelsize=9, length=3, width=0.5)
ax_left.set_ylabel("")
ax_left.set_xlabel("Permutation Feature Importance\n(% increase in prediction RMSE)", 
                   fontsize=10, 
                   labelpad=10)
ax_left.xaxis.set_major_formatter(PercentFormatter(xmax=100))

# Create legend
legend_elements = [Patch(facecolor=color, label=cat)
                  for cat, color in category_colors.items()]
ax_left.legend(handles=legend_elements,
              loc='lower right',
              frameon=False,
              fontsize=8,
              ncol=1)

# Panel B: SHAP Plot
# Remove 'control_itt_efficiency' from final plot
plot_features = [f for f in ALL_FEATURES if f != 'control_itt_efficiency']

# Create feature name mapping that matches Panel A's order
# Only create labels for those in DICT_COLUMN_LABELS
original_feature_names = [DICT_COLUMN_LABELS[x] for x in plot_features]

feature_name_to_idx = {name: idx for idx, name in enumerate(original_feature_names)}
# Filter ordered_features to those present in original_feature_names
ordered_features_filtered = [f for f in ordered_features if f in original_feature_names]

# Sort indices according to the ordered features (reversed as in original code)
sort_indices = [feature_name_to_idx[feature] for feature in ordered_features_filtered[::-1]]

# Extract indices from ALL_FEATURES corresponding to plot_features
plot_feature_indices = [plot_features.index(f) for f in plot_features]

# Map from original feature order to shap_values columns
shap_plot_indices = [plot_features.index(f) for f in plot_features]  # order as in plot_features
# Now reorder these indices according to sort_indices
shap_reorder_indices = [shap_plot_indices[i] for i in sort_indices]

shap_vals = shap_values.values[:, [plot_features.index(f) for f in plot_features]][:, shap_reorder_indices]
data_for_shap = background_data[:, [ALL_FEATURES.index(f) for f in plot_features]][:, shap_reorder_indices]

plt.sca(ax_right)
shap.summary_plot(
    shap_vals,
    data_for_shap,
    plot_type="dot",
    feature_names=ordered_features_filtered[::-1],
    show=False,
    alpha=0.5,
    sort=False
)

# Style SHAP plot
ax_right.set_yticklabels([])
ax_right.set_ylabel('')
ax_right.set_ylim(ax_left.get_ylim())
ax_right.tick_params(axis='both', which='major', labelsize=9)
ax_right.set_xlabel("SHAP Value\n(impact on prediction)", fontsize=10, labelpad=10)

# Add panel labels
ax_left.text(-0.1, 1.02, "A", transform=ax_left.transAxes, 
             fontsize=11, fontweight='bold')
ax_right.text(-0.1, 1.02, "B", transform=ax_right.transAxes,
              fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig('feature_importance_combined.pdf', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})

plt.savefig('feature_importance_combined.png', 
            dpi=300, 
            bbox_inches='tight',
            metadata={'Creator': 'Matplotlib'})
